### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="student_portuguese_performance",
    dataset_year="2008",
    domain_str="education",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C5TG7T",
    download_description="""
We get the portuguese data from the UCI repository. The math version is mostly duplicates from portuguese.

wget https://archive.ics.uci.edu/static/public/320/student+performance.zip && unzip student+performance.zip student.zip && unzip student.zip student-por.csv && rm student+performance.zip student.zip
mkdir -p local-data-warehouse/student_portuguese_performance && mv student-por.csv local-data-warehouse/student_portuguese_performance/
""",
    # References
    academic_reference_bibtex="""@article{silva2008using,
  title={Using data mining to predict secondary school student performance},
  author={Silva, Alice},
  year={2008}
}
""",
    academic_reference_bibtex_key="silva2008using",
    license="CC BY 4.0",
    data_tags=["IID"],
    curation_comments="""
We use the data from UCI.

- We drop G1 and G2 to simulate the task to predict the final grade from the base characteristic.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="G3",
    problem_type="regression",
    objective_metric_name="rmse",
)

## Preprocessing

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv(dataset_mold.path / "student-por.csv", sep=";")
print("Loaded data shape:", df.shape)

df = df.drop(columns=["G1", "G2"])
as_cat_type =  [
    "school",
    "sex",
    "address",
    "famsize",
    "Pstatus",
    "Mjob",
    "Fjob",
    "reason",
    "guardian",
    "schoolsup",
    "famsup",
    "paid",
    "activities",
    "nursery",
    "higher",
    "internet",
    "romantic",
]
df[as_cat_type] = df[as_cat_type].astype("category")
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

Loaded data shape: (649, 33)


## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 649
Columns: 31
Use sampling: False (sample size: 649)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['absences', 'age', 'Medu', 'Fjob', 'Mjob', 'freetime', 'Dalc', 'Fedu', 'famrel', 'health']
Rows remaining as candidates after top-10 filter: 2 (of 649)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,reason,guardian,traveltime,studytime,failures,schoolsup,famsup,paid,activities,nursery,higher,internet,romantic,famrel,freetime,goout,Dalc,Walc,health,absences,G3
0,MS,M,18,U,GT3,T,4,4,teacher,teacher,home,father,1,2,0,no,no,no,yes,no,yes,yes,no,3,2,4,1,4,2,4,19
1,GP,F,16,U,GT3,A,3,1,services,other,course,mother,1,2,0,no,yes,no,no,yes,yes,yes,no,2,3,3,2,2,4,2,12
2,MS,F,18,U,GT3,T,4,4,teacher,teacher,reputation,mother,2,2,0,no,no,no,yes,no,yes,yes,no,4,3,5,1,2,1,0,18
3,MS,M,16,R,LE3,A,4,4,at_home,other,home,mother,1,2,0,no,yes,no,no,yes,yes,no,no,5,3,2,1,3,2,5,11
4,GP,F,15,R,GT3,T,1,1,other,other,reputation,mother,1,2,0,yes,yes,no,no,no,yes,yes,yes,3,3,4,2,4,5,2,11


In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,school,category,0.0,0.0,2.0,"GP, MS"
1,sex,category,0.0,0.0,2.0,"F, M"
2,address,category,0.0,0.0,2.0,"U, R"
3,famsize,category,0.0,0.0,2.0,"GT3, LE3"
4,Pstatus,category,0.0,0.0,2.0,"T, A"
5,Mjob,category,0.0,0.0,5.0,"other, services, at_home, teacher, health"
6,Fjob,category,0.0,0.0,5.0,"other, services, at_home, teacher, health"
7,reason,category,0.0,0.0,4.0,"course, home, reputation, other"
8,guardian,category,0.0,0.0,3.0,"mother, father, other"
9,schoolsup,category,0.0,0.0,2.0,"no, yes"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
age,649.0,16.744222,1.218138,15.0,22.0
Medu,649.0,2.514638,1.134552,0.0,4.0
Fedu,649.0,2.306626,1.099931,0.0,4.0
traveltime,649.0,1.568567,0.748660,1.0,4.0
studytime,649.0,1.930663,0.829510,1.0,4.0
failures,649.0,0.221880,0.593235,0.0,3.0
famrel,649.0,3.930663,0.955717,1.0,5.0
freetime,649.0,3.180277,1.051093,1.0,5.0
goout,649.0,3.184900,1.175766,1.0,5.0
Dalc,649.0,1.502311,0.924834,1.0,5.0


In [7]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column     rank                          
Fjob       1          other    367  56.55
           2       services    181  27.89
           3        at_home     42   6.47
           4        teacher     36   5.55
           5         health     23   3.54
Mjob       1          other    258  39.75
           2       services    136  20.96
           3        at_home    135  20.80
           4        teacher     72  11.09
           5         health     48   7.40
Pstatus    1              T    569  87.67
           2              A     80  12.33
activities 1             no    334  51.46
           2            yes    315  48.54
address    1              U    452  69.65
           2              R    197  30.35
famsize    1            GT3    457  70.42
           2            LE3    192  29.58
famsup     1            yes    398  61.33
           2             no    251  38.67
guardian   1         mother    455  70.11
           2         father    153  23.57
           3          other     41   6.32
higher     1            yes    580  89.37
           2             no     69  10.63
internet   1            yes    498  76.73
           2             no    151  23.27
nursery    1            yes    521  80.28
           2             no    128  19.72
paid       1             no    610  93.99
           2            yes     39   6.01
reason     1         course    285  43.91
           2           home    149  22.96
           3     reputation    143  22.03
           4          other     72  11.09
romantic   1             no    410  63.17
           2            yes    239  36.83
school     1             GP    423  65.18
           2             MS    226  34.82
schoolsup  1             no    581  89.52
           2            yes     68  10.48
sex        1              F    383  59.01
           2              M    266  40.99

In [8]:
# Target Distribution
target_df

,y_missing_count,non_positive_pct,skew_y,skew_log,var_y,var_log,log_used,aic_exponential,aic_lognormal,dist_hint
0,0,2.31,-0.913,-4.264,10.437,0.194,log1p,4445.3,1020863.7,exponential


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=20, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...


Saving curated container to student_portuguese_performance/019d5dca-503e-7ea9-adac-f81a66c61295
019d5dca-503e-7ea9-adac-f81a66c61295
cf40946cf792d09641e68e1d870cf3d1996a617f7b34541550ead8724f43551b
